# Job Listing Prioritization for Career Changers: EDA/SDA 3 (Text features)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

**What do the listing descriptions say?** This notebook turns the description text of EDA_SDA_2's scope listings into **features**, and validates the text rules that hide listings, so `EDA_SDA_4` can test every feature (text included) and the app can hide what a career changer can't get. **It builds no matching models:** résumé-to-listing fit (keyword overlap, TF-IDF, BGE embeddings, RRF) is in `ML_Fit`, which uses this notebook's cleaned text and skills. A text feature is kept only if it adds information beyond the non-text features.

**For career changers,** the most important text features are what a listing asks of a newcomer (experience, education, licenses, training) and which of its skills carry over from other fields.

**Inputs:** EDA_SDA_2's scope table, plus the description text for those listings, loaded from the saved flattened parts (EDA_SDA_1's table keeps only description features, to fit in memory).

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the full file, how does tech compare with every field, and what's in the scope? (the scope cut, applied first) | Scope table and its descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? Additional cleaning, non-text checks by role bucket, ghost listings | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Text features, validated blockers, skills mapped to O*NET (no matching models) | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

**View:** listings with a description (77% of the scope; EDA_SDA_2 3.9). Text features are **unknown** for the rest, never "no", and "has a description" stays a standard confounder: listings without one close in about 3 days.

**Data rules:** aggregates only; never print descriptions, titles or sample rows, except in cells tagged `data-rows` (review cells for a teammate to check locally; cleared before every commit). Text stays local: no listing text goes to outside AI services. Clear outputs before committing.

### Clarifying questions
- What does description length actually capture: employer intent, or boilerplate?
- Does urgency wording, read in context rather than by keyword, relate to how long listings stay open? (EDA_SDA_1 left it here: its keyword flag misfired and showed no gap.)
- Can evergreen wording ("always accepting applications") identify ghost listings?
- Are EDA_SDA_1's career-change flags (years required, degree required, training and career-changer wording) accurate, and what do they miss? Two are urgent: "asks 5+ years" would hide 31% of open listings, and the degree flag fires on only 3–6% (EDA_SDA_2 3.11).
- Which requirements are blockers for everyone (hide the listing), and which depend on the applicant (degree, certification)?
- Which skills does each listing ask for, mapped to O*NET skills, so `ML_Fit` can match on skills rather than words?

### Roadmap

1. **Setup:** the scope table and its description text
2. **Description cleanup:** HTML to structured text, boilerplate removal, normalization and lemmas
3. **The blocker rules:** years of experience (required vs. preferred) and degree requirements
4. **Other text features:** evergreen wording, hiring-difficulty phrases, licenses and certifications, training and career-changer wording, urgency, remote, length
5. **Validation:** precision and recall of each rule on hand-checked samples
6. **Skills mapped to O*NET** (for `ML_Fit`)
7. **Final table:** validated blockers applied (hidden, never removed), saved for EDA_SDA_4 and `ML_Fit`
8. **Conclusion**


## Plan: text features to build

| Feature | Question | Notes |
|:--|:--|:--|
| **Description cleanup** (moved here from EDA_SDA_2 2b) | Strip HTML and boilerplate (equal-opportunity statements, benefits, "about us") | `desc_clean` and `desc_words_clean`; the base for every feature below and for `ML_Fit` |
| Description length | Is it employer intent, or just boilerplate? | Measured on `desc_clean` |
| Urgency wording | Does real urgency (in context, not "urgent care") go with shorter or longer windows? | EDA_SDA_1's keyword flag showed no gap; may mark hard-to-fill roles |
| Remote mentions | Does remote add beyond `contract_types`? | |
| **Evergreen wording** | Listings that collect applications instead of filling one job: a ghost-listing signal | Six phrase groups, each scored and checked on its own: **talent pool** ("join our talent community", "future opportunities"), **no specific opening** ("general application", "not for a specific position"), **always hiring** ("always accepting applications", "rolling basis"), **many openings in one ad** ("multiple openings", "various locations"), **pipeline labels** ("pipeline requisition", "evergreen requisition"), **staffing-style** ("on behalf of our client"); plus a generic-title check ("Software Engineer – All Levels"). Compared with EDA_SDA_2's likely-ghost flag (3.1): kept only if it adds to "old and from a quiet company" |
| Hiring-difficulty phrases | "Sign-on bonus", "hiring immediately", "relocation assistance", "will train" | Pairs with reposts and listing age |
| Years of experience required | "3+ years": is EDA_SDA_1's rule accurate? Required or preferred? In the field, or any experience? | The most common blocker for career changers |
| Education required | "Bachelor's required" vs. "or equivalent experience"; field-specific degrees | EDA_SDA_1's pattern only fires when "required" comes within six words of "bachelor's", so the most common wording ("Bachelor's degree in Computer Science or a related field") never matches: it flags only 3–6% of listings. Rebuilt in section 3, with requirements and preferred sections told apart |
| **Clearance, sponsorship, travel, shifts** (added 2026-10-10) | Security clearance required; "no visa sponsorship"; travel percentage; shift or on-call work; contract-to-hire | Clearance and sponsorship are hard blockers for some career changers: hidden per applicant in the app, like a missing degree |
| Licenses and certifications required | "Security+", "CCNA", "CISSP" and the like | A required credential the career changer lacks is a blocker |
| Training and career-changer wording | "Will train", "paid training", "career changers welcome", "transferable skills" | Validates EDA_SDA_1's flags: a team member checks a random sample of matches locally (never shared with AI tools) |
| **Hard-blocker rules** | Which years-asked threshold and which required credentials hide a listing from everyone, per role bucket? | Validated here, then applied in this notebook's final table, with a count of the listings each rule hides (hidden in the app, never removed) |
| Required vs. preferred qualifications | Split on headings like "Requirements" and "Nice to have" | Part of fit and gap analysis in `ML_Fit` |
| **Skills, mapped to O*NET** | Extracted with spaCy (PredictLeads `tags` are sparse) and mapped to O*NET skills | `ML_Fit` matches résumés to listings on these skills, so "transferable" doesn't depend on the old field's vocabulary |

**Moved to `ML_Fit`:** role similarity, TF-IDF and embeddings for matching. **Moved to `ML_Rank`:** TF-IDF terms that predict how fast a listing closes (a text input to the timing model). Topic modeling stays a stretch goal.

## Plan: preprocessing

Course standard (MLforTexts): normalize text (lowercase, regex to keep letters), remove NLTK stopwords, lemmatize with spaCy. Advanced: remove boilerplate where headings make it detectable. Any vocabulary learned from the text is fit on the training months only.

**Evaluation (course standard):** each validated rule reports precision and recall on a hand-checked random sample (checked locally by a teammate), against the keyword rule from EDA_SDA_1 as the baseline.


## 1. Setup

In [ ]:
# 1a Libraries, paths and the shared code: src/text_features.py does the cleanup and the text rules batch by batch,
# so the full scope's descriptions never sit in memory at once.
import gc
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, INK, MUTED, CLEAN_DIR, PART2_TABLE, PART3_TABLE,
    bar_h, finish, show, show_shaded, use_chart_style,
)
from src import text_features as tf
use_chart_style()

TRAIN_END = "2026-07"  # anything learned from the text (template lines) is learned on April–July only, ML_Rank's training months
DESC_PATH = CLEAN_DIR / "scope_descriptions.parquet"  # EDA_SDA_1 8d: id, description (HTML)
FEATURES_PATH = CLEAN_DIR / "eda3_text_features.parquet"  # every listing's text features (cache; delete to rebuild)
CLEAN_PATH = CLEAN_DIR / "eda3_desc_clean.parquet"  # every listing's cleaned text, for ML_Fit
LEMMA_PATH = CLEAN_DIR / "eda3_lemmas_sample.parquet"  # lemmas for a stratified sample (2c)
REMOVED_PATH = CLEAN_DIR / "eda3_removed_words.json"  # words removed per boilerplate reason (2b)
VALIDATION_DIR = CLEAN_DIR / "eda3_validation"  # hand-check samples (section 5); local only, like all of data/
TEMPLATE_SAMPLE = 120_000  # listings used to learn shared template lines (training months, with a description)
LEMMA_SAMPLE = 150_000  # listings lemmatized with spaCy (stratified by role bucket)
_t0 = time.time()

In [ ]:
# 1b Load only the columns this notebook needs from EDA_SDA_2's scope table (the text itself is read batch by batch
# in section 2). Only counts are shown.
COLS = ["id", "company_id", "first_seen", "month", "role_bucket", "tripleten_program", "has_desc", "live", "days_open",
        "volume_band", "source_type", "likely_ghost", "stale", "cc_visible", "level", "is_repost", "hidden_occupation", "career_open",
        "years_required", "degree_required", "equivalent_experience", "entry_wording", "changer_wording", "urgency", "mentions_remote"]
jobs = pd.read_parquet(PART2_TABLE, columns=COLS)
jobs["month"] = jobs.month.astype(str)
meta = jobs.set_index("id")[["company_id", "month"]]
show("1b The scope table and its text", pd.DataFrame({"value": {
    "listings (EDA_SDA_2 scope table)": f"{len(jobs):,}",
    "with a description (has_desc)": f"{jobs.has_desc.mean():.1%}",
    "descriptions file": f"{DESC_PATH.name}: {pq.ParquetFile(DESC_PATH).metadata.num_rows:,} rows, {DESC_PATH.stat().st_size / 1e9:.2f} GB",
    "first seen in the training months (April–July)": f"{(jobs.month <= TRAIN_END).sum():,}",
}}))

## 2. Description cleanup

**Why:** every text feature, and `ML_Fit`'s matching, should read what the listing asks for, not the legal and benefits text most listings share. At full scale the descriptions are several GB of text, so **the cleanup runs batch by batch** (`src/text_features.py`): each batch is cleaned, its features computed and both written to disk, then dropped from memory.

| Step | What | Output |
|:--|:--|:--|
| 2a | HTML to text, one line per paragraph, list item or heading (long unbroken text split into sentences) | Lines per listing |
| 2b | Remove boilerplate line by line: known patterns (equal opportunity and legal, benefits, application logistics), "About the company" sections, and text shared word for word across 5+ employers (learned on a sample of April–July listings; lines with a requirement cue are never cut) | `desc_clean` (`eda3_desc_clean.parquet`), `words_clean` |
| 2c | Course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas, on a stratified sample | `desc_lemmas` (for word counts and TF-IDF later) |

The rules in sections 3–4 run on the cleaned text, which keeps case, numbers and punctuation ("3+ years", "B.S.", "Security+"), and tell **requirement lines from preferred ones** using the headings kept in 2a.

In [ ]:
# 2a–2b Learn shared template lines on a sample of training-month listings, then clean every description and compute
# its text features, batch by batch on several processes (cached: delete eda3_text_features.parquet to rebuild).
have = jobs[jobs.has_desc & (jobs.month <= TRAIN_END)]
sample_ids = set(have.sample(min(TEMPLATE_SAMPLE, len(have)), random_state=42).id)
if tf.cache_valid(FEATURES_PATH, DESC_PATH, REMOVED_PATH, CLEAN_PATH):  # an unreadable file (a killed run) means rebuild
    removed = json.loads(REMOVED_PATH.read_text())
    print(f"loaded the text features from {FEATURES_PATH.name} (delete it to rebuild 2a–2b)")
else:
    templates = tf.learn_templates(DESC_PATH, sample_ids, meta.company_id)  # streams batches on several processes; keeps only line keys
    print(f"shared template lines learned on {len(sample_ids):,} April–July listings: {len(templates):,}")
    start = time.time()
    removed = tf.build(DESC_PATH, meta, templates, FEATURES_PATH, CLEAN_PATH,
                       progress=lambda n, k: print(f"  batch {n + 1}: {k:,} listings, {time.time() - start:.0f}s", end="\r"))
    REMOVED_PATH.write_text(json.dumps(removed))
    print(f"\ncleaned and featurized every description in {(time.time() - start) / 60:.1f} min")
text = pd.read_parquet(FEATURES_PATH, dtype_backend="numpy_nullable")  # blanks stay unknown (NA), not None in object columns
jobs = jobs.merge(text, on="id", how="left", suffixes=("_eda1", ""))
has_text = jobs.words_raw.fillna(0) > 0
by_reason = pd.Series(removed, dtype=float).sort_values(ascending=False)
show("2b Words removed as boilerplate (listings with a description)", pd.DataFrame({
    "words removed": by_reason,
    "share of all description words": by_reason / jobs.words_raw.sum(),
}).round(3))
src = jobs[has_text].groupby("source_type")
show("2b Share of each description kept after cleanup, by source (median per listing)", pd.DataFrame({
    "median words before": src.words_raw.median(), "median words after": src.words_clean.median(),
    "median share kept": src.apply(lambda g: (g.words_clean / g.words_raw).median(), include_groups=False),
    "cleaned to nothing": src.apply(lambda g: g.words_clean.eq(0).mean(), include_groups=False),
    "listings": src.size(),
}).round(3))

**What the data shows**
- (fill in after the run)

In [ ]:
# 2b review: a few random listings after cleanup, for a teammate to check locally that 2b keeps the requirements.
# Shows real listing text. Tagged "data-rows": clear this output before committing.
clean_sample = pq.ParquetFile(CLEAN_PATH).read_row_group(0).to_pandas().sample(3, random_state=7)
for t in clean_sample.desc_clean:
    print("=" * 100, "\n", t[:2500])

In [ ]:
# 2c Normalize and lemmatize (course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas) on a stratified
# sample, since spaCy on every listing would take hours. desc_lemmas feeds word counts and TF-IDF in ML_Fit and ML_Rank.
import nltk
import spacy
try:
    STOP = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)  # one-time download of NLTK's word list (no project data leaves the laptop)
    STOP = set(nltk.corpus.stopwords.words("english"))

if LEMMA_PATH.exists():
    lem = pd.read_parquet(LEMMA_PATH)
    print(f"loaded {len(lem):,} lemmatized listings from {LEMMA_PATH.name}")
else:
    pool = jobs[has_text]
    lem_ids = set(pool.groupby("role_bucket").sample(frac=min(LEMMA_SAMPLE / len(pool), 1), random_state=42).id)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # the tagger and lemmatizer are all that's needed
    parts = []
    for batch in pq.ParquetFile(CLEAN_PATH).iter_batches(batch_size=20_000):
        d = batch.to_pandas()
        d = d[d.id.isin(lem_ids)]
        lemmas = [" ".join(t.lemma_.lower() for t in doc if t.is_alpha and t.lemma_.lower() not in STOP)
                  for doc in nlp.pipe(d.desc_clean.str.replace("\n", ". ").str[:20000], batch_size=64)]
        parts.append(pd.DataFrame({"id": d.id.to_numpy(), "desc_lemmas": lemmas}))
    lem = pd.concat(parts, ignore_index=True)
    lem.to_parquet(LEMMA_PATH, index=False)
    print(f"saved {len(lem):,} lemmatized listings to {LEMMA_PATH.name}")
words_lem = lem.desc_lemmas.str.split().str.len()
show("2c Words per description at each step (listings with a description)", pd.DataFrame({
    "after HTML removal": jobs.words_raw[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "after boilerplate removal": jobs.words_clean[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "lemmas, stopwords removed (sample)": words_lem.describe(percentiles=[0.1, 0.5, 0.9]),
}).drop(index="count").round(0))

In [ ]:
# 2 Chart: words per description at each cleanup step (left), and what boilerplate removal takes out (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
bins = np.logspace(0, np.log10(max(jobs.words_raw.max(), 10)), 40)
for vals, label, color in [(jobs.words_raw[has_text], "after HTML removal", MUTED), (jobs.words_clean[has_text], "after boilerplate removal", BLUE),
                           (words_lem, "lemmas, stopwords removed (sample)", ORANGE)]:
    ax1.hist(vals.clip(lower=1), bins=bins, histtype="step", lw=1.8, color=color, label=label, density=True)
ax1.set(xscale="log", title="Words per description at each step", xlabel="words (log scale)", ylabel="share of listings (density)")
ax1.legend(fontsize=8, frameon=False)
bar_h(ax2, 100 * by_reason / jobs.words_raw.sum(), "Share of all description words removed", "% of words", "{:.1f}%")
finish()

**What the data shows**
- (fill in after the run)

## 3. The blocker rules: years of experience and degree

EDA_SDA_1's rules read every line the same way: "3+ years required" and "5 years preferred" both count, and its degree rule only fires when "required" comes within six words of "bachelor's", so the most common wording ("Bachelor's degree in Computer Science or a related field") never matched (3–6% of listings). Here both are rebuilt on the cleaned text, with **requirement lines told apart from preferred ones** (from the headings kept in 2a, and lines that say "preferred" or "a plus"):

- **Years:** `years_asked` (any line, like EDA_SDA_1), `years_required` (requirement lines only), `years_preferred`. Also reads "minimum of two years" and number words.
- **Degree:** `degree_required` (a degree named on a requirement line), `degree_preferred`, `degree_equivalent` ("or equivalent experience", "in lieu of a degree"), `degree_level` (the highest level required).

The question for the app: **which rule can hide a listing from everyone** (too many years required), and which hide it only per applicant (a degree they don't have). Nothing hides until section 5 validates the rule.

In [ ]:
# 3a Years of experience: the new rules vs. EDA_SDA_1's, and how many listings each would hide at 5+ years
def share(mask):
    """Share of rows where mask is true; a blank (no years stated) counts as false, so every share is over all rows."""
    return mask.fillna(False).astype(bool).mean()

d = jobs[has_text]
y_old, y_new = d.years_required_eda1.astype(float), d.years_asked.astype(float)
agree = pd.Series({
    "both say no years": share(y_old.isna() & y_new.isna()),
    "same number": share(y_old == y_new),
    "new rule finds more years (or finds years where EDA_SDA_1 found none)": share(y_new > y_old.fillna(0)),
    "new rule finds fewer (or none where EDA_SDA_1 found some)": share(y_new.fillna(0) < y_old),
}, name="share of listings with a description")
show("3a EDA_SDA_1's years rule vs. the new one (any line)", agree.to_frame())
open_now = d.live
y_req, y_pref = d.years_required.astype(float), d.years_preferred.astype(float)
rules = {
    "EDA_SDA_1: 5+ years anywhere": (y_old >= 5),
    "new: 5+ years anywhere": (y_new >= 5),
    "new: 5+ years on a requirement line": (y_req >= 5),
    "new: 5+ years only on preferred lines": (y_pref >= 5) & ~(y_req >= 5),
}
show("3a The 5+ years rule: share of listings with a description, and of open ones (what it would hide)", pd.DataFrame({
    "all listings": {k: share(v) for k, v in rules.items()},
    "open on September 30": {k: share(v[open_now]) for k, v in rules.items()},
}).round(3))
by_bucket = d.assign(y_new=y_new, y_req=y_req, y_old=y_old).groupby("role_bucket").agg(
    states_years=("y_new", lambda s: s.notna().mean()),
    median_required=("y_req", "median"),
    required_5plus=("y_req", lambda s: (s >= 5).mean()),
    required_2_or_fewer=("y_req", lambda s: (s <= 2).mean()),
    eda1_5plus=("y_old", lambda s: (s >= 5).mean()),
    listings=("id", "size"),
).sort_values("required_5plus")
show("3a Years required by role bucket (listings with a description)", by_bucket.round(3))

In [ ]:
# 3b Degree: the new rule vs. EDA_SDA_1's, with "or equivalent experience" and the level asked
d = jobs[has_text]
show("3b Degree rules: share of listings with a description", pd.Series({
    "EDA_SDA_1: degree required": d.degree_required_eda1.mean(),
    "new: a degree mentioned anywhere": d.degree_mentioned.mean(),
    "new: degree on a requirement line": d.degree_required.mean(),
    "new: degree only on preferred lines": (d.degree_preferred & ~d.degree_required).mean(),
    "new: or equivalent experience accepted": d.degree_equivalent.mean(),
    "new: degree required, no equivalent (the per-applicant blocker)": (d.degree_required & ~d.degree_equivalent).mean(),
}, name="share").round(3).to_frame())
show("3b Highest degree level on a requirement line", d.degree_level.value_counts(normalize=True, dropna=False).rename("share").round(3).to_frame())
show("3b Degree required (no equivalent) by role bucket", d.groupby("role_bucket").apply(
    lambda g: pd.Series({"degree required": g.degree_required.mean(), "no equivalent": (g.degree_required & ~g.degree_equivalent).mean(),
                         "bachelor's or higher": g.degree_level.isin(["bachelor", "master", "phd"]).mean()}),
    include_groups=False).sort_values("no equivalent").round(3))

In [ ]:
# 3 Chart: share asking 5+ years on a requirement line vs. EDA_SDA_1's rule, by role bucket
fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(by_bucket))
ax.barh(y - 0.2, 100 * by_bucket.eda1_5plus, height=0.4, color=MUTED, label="EDA_SDA_1: 5+ years anywhere")
ax.barh(y + 0.2, 100 * by_bucket.required_5plus, height=0.4, color=BLUE, label="new: 5+ years on a requirement line")
ax.set(yticks=y, yticklabels=by_bucket.index, xlabel="% of listings with a description", title="Who asks 5+ years: old rule vs. requirement lines only")
ax.legend(fontsize=8, frameon=False)
finish()

**What the data shows**
- (fill in after the run)

## 4. Other text features

Each feature is a keyword rule on the cleaned text (`src/text_features.py`), validated in section 5 before it's used to hide anything. For each: how often it fires, and whether it relates to how fast listings close **beyond the confounders** (the 7-day close rate's gap with vs. without the feature, within company volume band × source type cells, on original listings with a full week of follow-up; EDA_SDA_4 tests the survivors properly).

| Group | Features |
|:--|:--|
| Blockers for some applicants | `clearance_required` (vs. `clearance_obtainable`), `no_sponsorship`, `us_citizen_required`, `cert_required` and named certifications, `shift_or_oncall`, `travel_pct`, `contract_to_hire` |
| What a newcomer is offered | `entry_wording`, `changer_wording` (now also bootcamp graduates, self-taught) |
| Hiring difficulty and urgency | `sign_on_bonus`, `hiring_immediately`, `relocation_offered`, `urgency_context` ("urgent care" no longer counts) |
| Evergreen wording (ghost signal) | six phrase groups and `evergreen_any`, compared with EDA_SDA_2's likely-ghost flag |
| Remote and length | `remote_mention` (minus "remote patient monitoring" and the like), `words_clean` |

In [ ]:
# 4a How often each text feature fires, overall and by role bucket (listings with a description)
FLAGS = [c for c in tf.FLAG_COLS if c not in ("degree_mentioned", "degree_required", "degree_preferred", "degree_equivalent")]
d = jobs[has_text]
rates = d[FLAGS].astype(float).mean().sort_values(ascending=False)
show("4a Share of listings with a description where each feature fires", rates.rename("share").round(4).to_frame())
show_shaded("4a Text features by role bucket (share of listings; features firing on 0.5%+)",
            d.groupby("role_bucket")[[c for c in FLAGS if rates[c] >= 0.005]].mean().T.round(3))

In [ ]:
# 4b Does each feature go with faster or slower closing, beyond the confounders? Original listings with a full week of
# follow-up; the 7-day close rate with vs. without the feature, raw and within volume band × source type cells
# (weighted by the cell's listings). A first look: EDA_SDA_4's adjusted model decides.
cut = pd.Timestamp("2026-09-23", tz=jobs.first_seen.dt.tz)
o = jobs[has_text & ~jobs.is_repost & (jobs.first_seen <= cut)].copy()
o["closed_7d"] = (~o.live & (o.days_open <= 7)).astype(float)
o["cell"] = o.volume_band.astype(str) + " | " + o.source_type.astype(str)

def within_gap(flag):
    g = o.groupby(["cell", o[flag].astype(bool)]).closed_7d.agg(["mean", "size"]).unstack()
    g = g.dropna()
    g = g[(g[("size", True)] >= 200) & (g[("size", False)] >= 200)]
    if g.empty:
        return np.nan
    w = g[("size", True)] + g[("size", False)]
    return float(((g[("mean", True)] - g[("mean", False)]) * w).sum() / w.sum())

rows = {}
for f in FLAGS + ["degree_required", "degree_equivalent"]:
    on = o[f].astype(bool)
    if on.sum() < 500:
        continue
    rows[f] = {"share": on.mean(), "closed in 7 days (with)": o.closed_7d[on].mean(), "closed in 7 days (without)": o.closed_7d[~on].mean(),
               "raw gap (points)": 100 * (o.closed_7d[on].mean() - o.closed_7d[~on].mean()), "gap within confounders (points)": 100 * within_gap(f)}
show("4b Text features vs. closing within 7 days (original listings with a description and a full week of follow-up)",
     pd.DataFrame(rows).T.sort_values("gap within confounders (points)").round(3) if rows
     else pd.DataFrame({"note": ["no feature fires on 500+ of these listings"]}))

In [ ]:
# 4c Evergreen wording vs. EDA_SDA_2's ghost signals: does it add to "old and from a quiet company"?
d = jobs[has_text]
grp = pd.Series(np.where(d.likely_ghost, "likely ghost", np.where(d.stale, "open 90+ days, company still posting",
                np.where(d.live, "open, 90 days or younger", "closed"))), index=d.index)
show("4c Evergreen wording by ghost status (share of listings with a description)",
     d.groupby(grp)[tf.EVERGREEN + ["evergreen_any"]].mean().round(4))

**What the data shows**
- (fill in after the run)

## 5. Validation

Each rule that could hide a listing, or that feeds the model, gets a **hand-checked sample**: 60 listings where it fires and 60 where it doesn't, stratified by role bucket, written to `data/clean/eda3_validation/<rule>.csv` (local only) with the line that triggered it. A teammate marks each row `correct` = 1 or 0 locally; the notebook reads the marks back and reports **precision** (of the matches, how many are right) and **recall** (of the misses, how many should have matched), against EDA_SDA_1's rule where one exists. **A rule hides listings only if its precision is 90%+.** Listing text never leaves the laptop.

In [ ]:
# 5a Write the hand-check samples (only if they don't exist yet, so marks are never overwritten). Shows counts only.
RULES = ["years_required_5plus", "degree_required", "degree_equivalent", "clearance_required", "no_sponsorship", "cert_required",
         "entry_wording", "changer_wording", "evergreen_any", "urgency_context"]
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
d = jobs[has_text].assign(years_required_5plus=jobs.years_required.astype(float) >= 5)
written = {}
for rule in RULES:
    path = VALIDATION_DIR / f"{rule}.csv"
    if path.exists():
        continue
    picks = []
    for value in (True, False):  # about 60 matches and 60 non-matches, spread across role buckets by their size
        pool = d[d[rule].fillna(False).astype(bool) == value]
        if len(pool):
            picks.append(pool.groupby("role_bucket").sample(frac=min(60 / len(pool), 1), random_state=42).assign(fires=value))
    sample = pd.concat(picks)
    sample.assign(rule=rule, correct="")[["id", "rule", "fires", "role_bucket", "correct"]].to_csv(path, index=False)
    written[rule] = len(sample)
show("5a Hand-check samples written (open each CSV locally with the listing text from eda3_desc_clean.parquet)",
     pd.Series(written, name="rows").to_frame() if written else pd.DataFrame({"note": ["all samples already exist"]}))

In [ ]:
# 5b Read the marks back: precision of each rule (and recall from the non-matches), once a teammate has filled them in
res = {}
for path in sorted(VALIDATION_DIR.glob("*.csv")):
    v = pd.read_csv(path)
    v = v[v.correct.isin([0, 1, "0", "1"])].assign(correct=lambda x: x.correct.astype(int))
    if v.empty:
        continue
    hit, miss = v[v.fires], v[~v.fires]
    res[path.stem] = {"marked": len(v), "precision": hit.correct.mean() if len(hit) else np.nan,
                      "share of non-matches that are right": miss.correct.mean() if len(miss) else np.nan,
                      "hides listings (precision 90%+)": bool(len(hit) and hit.correct.mean() >= 0.9)}
show("5b Validation results", pd.DataFrame(res).T if res else pd.DataFrame({"note": ["no marks yet: fill in the CSVs in eda3_validation/, then rerun"]}))

**What the data shows**
- (fill in after the run)

## 6. Skills mapped to O*NET

**Waiting for the O*NET download** (terms of use first; `analysis/ANALYSIS.md`). Then: skills extracted from the requirement lines (spaCy noun phrases plus a skills vocabulary), mapped to O*NET skills for `ML_Fit`'s skills-based matching. The cleaned text it needs is already saved (`eda3_desc_clean.parquet`).

## 7. Final table

One row per scope listing (`id`), the text features from sections 3–4, and **candidate blocker flags** for the app. A flag only hides listings once its rule passes validation (section 5); until then it's saved, not applied. Blank means unknown: the listing has no description. Saved as `text_table.parquet` for EDA_SDA_4 (joined by `id`) and `ML_Fit`.

In [ ]:
# 7a Candidate blockers (applied in the app only after validation) and what each would hide among open listings
out = jobs[["id"] + tf.FEATURE_COLS].copy()
known = jobs.words_raw.fillna(0) > 0
out["blocker_years_5plus"] = (jobs.years_required.astype(float) >= 5).where(known)  # everyone, if validated (no years stated = False)
out["blocker_degree_no_equivalent"] = (jobs.degree_required & ~jobs.degree_equivalent).where(known)  # per applicant
out["blocker_clearance"] = jobs.clearance_required.where(known)                 # per applicant
out["blocker_no_sponsorship"] = jobs.no_sponsorship.where(known)                # per applicant (needs sponsorship)
out["blocker_cert_required"] = jobs.cert_required.where(known)                  # per applicant
BLOCKERS = [c for c in out if c.startswith("blocker_")]
show("7a Candidate blockers: share of listings with a description, and of open ones", pd.DataFrame({
    "all listings": out.loc[known, BLOCKERS].astype(float).mean(),
    "open on September 30": out.loc[known & jobs.live, BLOCKERS].astype(float).mean(),
}).round(3))
out.to_parquet(PART3_TABLE, index=False)
show("7a The final text table", pd.DataFrame({"value": {
    "listings": f"{len(out):,}", "columns": out.shape[1], "with text features": f"{known.mean():.1%}",
    "saved": f"{PART3_TABLE.name} ({PART3_TABLE.stat().st_size / 1e9:.2f} GB)",
}}))
print(f"[EDA_SDA_3 done in {(time.time() - _t0) / 60:.1f} min]")

## 8. Conclusion

- (fill in after the run: the cleanup, the rebuilt years and degree rules, the other text features, what validation shows, and what goes to EDA_SDA_4 and ML_Fit)

**Handoff:** `text_table.parquet` (text features and candidate blockers, joined by `id`) for EDA_SDA_4 and `ML_Fit`; `eda3_desc_clean.parquet` (cleaned text) and `eda3_lemmas_sample.parquet` for `ML_Fit`.